# 12.4 - LangChain Chains
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
Real apps need many steps: classify then extract, retrieve then answer, draft then critique. Chains
let you wire runnables (prompt | model | parser) together with the pipe operator instead of writing
nested callbacks.
## 2. Why Does This Matter?
Without chains you are back to manual plumbing (Unit 12.1). Chains compose reusable building blocks
into one callable pipeline that is easy to inspect and reason about.
## 3. Prerequisites
- Units 12.1-12.3 (manual, models/prompts, parsers)
- Python function composition
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Compose `prompt | model | parser` with LCEL
- Build two independent chains (summarize, translate) and a two-step compose chain
- Inspect a chain structure with `.get_graph().print_ascii()`
- Recognise legacy `LLMChain` and know why we avoid it
## 5. Mental Model
A chain is a Unix pipe for data: each step transforms its input and hands it on. Data flows left to
right; a chain is lazy — it does nothing until `.invoke()`.

```text
input -> prompt -> model -> parser -> output
  |        |         |        |
 text    messages  AIMessage  typed


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq


def chat_invoke(messages, temperature=0.0, model=GROQ_MODEL):
    if not os.environ.get("GROQ_API_KEY"):
        last = str(getattr(messages, "content", messages[-1].content))[:40] if hasattr(messages, "content") else "input"
        return AIMessage(content=f"mock: processed [{last}]")
    try:
        return ChatGroq(model=model, temperature=temperature).invoke(messages)
    except Exception as e:
        return AIMessage(content=f"[llm-error: {type(e).__name__}]")


safe_model = RunnableLambda(lambda m: chat_invoke(m))


## 7. The Core Pattern: prompt | model | parser
LCEL chains are built with `|`. This is our first real chain: template -> model -> string parser.

In [3]:
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize the following in one sentence.\n\nText: {text}")
summarizer = summarize_prompt | safe_model | StrOutputParser()

out = summarizer.invoke({"text": "LangChain lets you compose prompts, models and parsers with the "
                                  "pipe operator, which keeps multi-step pipelines readable."})
print(out)


LangChain uses the pipe operator to compose prompts, models, and parsers, making multi‑step pipelines readable.


## 8. A Second Independent Chain: Translate
The same building blocks, a different template. Each chain is a self-contained runnable.

In [4]:
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this to {language}: {text}")
translator = translate_prompt | safe_model | StrOutputParser()

print(translator.invoke({"language": "Spanish", "text": "Good morning, team."}))


Buenos días, equipo.


## 9. Chain Composition: Draft Then Critique
Only one prompter/model isn't enough. Compose a second stage that receives the *output* of the first.
We adapt the string into the dict the next prompt expects with a lambda.

In [5]:
draft_prompt = ChatPromptTemplate.from_template(
    "Write a two-sentence feature blurb about: {topic}")
critic_prompt = ChatPromptTemplate.from_template(
    "Critique the blurb below. Give one specific improvement.\n\nBlurb: {blurb}")

draft = draft_prompt | safe_model | StrOutputParser()
critic = critic_prompt | safe_model | StrOutputParser()

chain2 = draft | (lambda b: {"blurb": b}) | critic
print(chain2.invoke({"topic": "a spellchecker that runs fully offline"}))


**Critique**  
The blurb is clear and concise, but it reads like a generic elevator pitch. It tells the reader what the product does (offline, fast, privacy‑friendly) but doesn’t explain *why* that matters or how it’s better than other spellcheckers. The lack of a concrete benefit or unique feature makes the promise feel vague and less memorable.

**One specific improvement**  
Add a concrete, user‑centric benefit that quantifies the value. For example:

> “Introducing the ultimate offline spellchecker—accurate, lightning‑fast, and never asks for an internet connection. Save up to 30 % of your typing time with AI‑powered suggestions, protect your privacy, and keep your workflow uninterrupted, no matter where you are.”

This tweak turns an abstract claim into a tangible advantage that readers can immediately relate to.


## 10. Visualize the Chain As a Graph
`.get_graph().print_ascii()` shows the internal DAG — the exact same mental model as Phase 13
(LangGraph). This is how you debug a chain's shape.

In [6]:
def show_graph(chain, name):
    g = chain.get_graph()
    print(f"--- {name} graph ---")
    try:
        g.print_ascii()          # needs grandalf; falls back if absent
    except Exception:
        print(g.draw_mermaid())


show_graph(summarizer, "summarizer")


--- summarizer graph ---
---
config:
  flowchart:
    curve: linear
---
graph TD;
	PromptInput([PromptInput]):::first
	ChatPromptTemplate(ChatPromptTemplate)
	Lambda(Lambda)
	StrOutputParser(StrOutputParser)
	StrOutputParserOutput([StrOutputParserOutput]):::last
	PromptInput --> ChatPromptTemplate;
	ChatPromptTemplate --> Lambda;
	StrOutputParser --> StrOutputParserOutput;
	Lambda --> StrOutputParser;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [7]:
show_graph(chain2, "chain2 (draft -> critic)")


--- chain2 (draft -> critic) graph ---
---
config:
  flowchart:
    curve: linear
---
graph TD;
	PromptInput([PromptInput]):::first
	ChatPromptTemplate_1(ChatPromptTemplate)
	Lambda_1(Lambda)
	StrOutputParser_1(StrOutputParser)
	Lambda_2(Lambda)
	ChatPromptTemplate_2(ChatPromptTemplate)
	Lambda_3(Lambda)
	StrOutputParser_2(StrOutputParser)
	StrOutputParserOutput([StrOutputParserOutput]):::last
	PromptInput --> ChatPromptTemplate_1;
	ChatPromptTemplate_1 --> Lambda_1;
	Lambda_1 --> StrOutputParser_1;
	StrOutputParser_1 --> Lambda_2;
	Lambda_2 --> ChatPromptTemplate_2;
	ChatPromptTemplate_2 --> Lambda_3;
	StrOutputParser_2 --> StrOutputParserOutput;
	Lambda_3 --> StrOutputParser_2;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 11. RunnablePassthrough: Pass a Field Through
Sometimes you want to keep the original input alongside a transformed value. `RunnablePassthrough`
carries the input dict forward (or, in `assign`, adds a computed key).

In [8]:
full = RunnableParallel(
    summary=summarizer,
    original=RunnablePassthrough(),
)
result = full.invoke({"text": "RAG combines retrieval with generation to ground answers in evidence."})
print("summary  :", result["summary"])
print("original :", result["original"]["text"])


summary  : RAG blends retrieval and generation to anchor its answers in evidence.
original : RAG combines retrieval with generation to ground answers in evidence.


## 12. Legacy LLMChain (Briefly)
`LLMChain` was the old global `chain` object combining a prompt and a model in one class. It still
imports in LangChain 1.x but is **legacy**: it is replaced by the `|` composition you just used. We
show it only so you recognise it in old code — we do not build with it.

In [9]:
try:
    from langchain.chains import LLMChain
    legacy = LLMChain(llm=ChatGroq(model=GROQ_MODEL, temperature=0.0), prompt=draft_prompt)
    print("legacy LLMChain constructed. Prefer: prompt | model | parser instead.")
except Exception as e:
    print("LLMChain import/setup skipped:", type(e).__name__)


LLMChain import/setup skipped: ModuleNotFoundError




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Common Mistakes (applied)

- Deeply nested chains that are hard to debug — keep 3-5 steps.
- Not testing each step independently before composing.
- Forgetting chains are lazy (they run only on `.invoke`).
- Wrong dict key passed into the next prompt (the lambda adapter is the fix).

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| Wrong output type | Missing parser | Add `StrOutputParser` at the end |
| `RunnableLambda` error | Function does not accept the dict | Take `{"key": value}` and return what next needs |
| Chain slow | Sequential steps | Use `RunnableParallel` for independent steps |
| Data lost mid-chain | Wrong key extraction | Adapter lambda or `RunnablePassthrough` |

### Best Practices (applied)

- Test each step alone before composing.
- Keep chains shallow.
- Inspect with `.get_graph().print_ascii()`.

### Hands-On Practice

1. **Basic:** Add a third step that uppercases the critique.
2. **Guided:** Build classify-then-extract as a two-prompt chain.
3. **Independent:** Use `RunnableParallel` to run `summarizer` and `translator` on the same text at once.
4. **Realistic:** Build a 4-step chain: extract entities -> classify -> rank -> format report.
5. **Challenge:** Rebuild `chain2` with raw Python functions and compare `.print_ascii()` vs debugging.

### Exit Criteria

- You can compose multiple steps into a chain.
- You can debug intermediate outputs.
- You can use `RunnableParallel` for concurrent operations.
